# K-Means Clustering

K-Means is the most widely used clustering algorithm. Given a dataset of $n$ points and a target number of clusters $K$, it partitions the data into $K$ groups by minimizing the total within-cluster spread.

---

## Table of Contents
1. [Intuition — What is Clustering?](#1-intuition)
2. [The Objective Function](#2-objective-function)
3. [Lloyd's Algorithm](#3-lloyds-algorithm)
4. [Mathematical Proof of Convergence](#4-convergence)
5. [K-Means++ Initialization](#5-kmeans-plus-plus)
6. [The Elbow Method](#6-elbow-method)
7. [Limitations of K-Means](#7-limitations)
8. [Implementation from Scratch](#8-from-scratch)
9. [sklearn K-Means on Real Data](#9-sklearn)
10. [Summary](#10-summary)

---
## 1. Intuition — What is Clustering?

Clustering is an **unsupervised** task: given data with no labels, find natural groupings.

K-Means answers the question: *"If I must divide these points into K groups, what is the best assignment?"*

**Best** here means: points in the same cluster should be close to each other, and the center of each cluster (the **centroid**) should be as representative of its members as possible.

### Everyday Analogy

Imagine placing $K$ fire stations in a city to minimize the average travel time to any point. Each point (home/building) belongs to the nearest station. After assigning points, you move each station to the center of its assigned area. Repeat until stable — that is K-Means.

---
## 2. The Objective Function

K-Means minimizes the **Within-Cluster Sum of Squares (WCSS)**, also called **inertia**:

$$J = \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

where:
- $C_k$ = set of points assigned to cluster $k$
- $\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i$ = centroid of cluster $k$
- $\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$ = squared Euclidean distance from point to centroid

### Why Squared Distance?

- Mathematically convenient: the minimizer of $\sum (x_i - c)^2$ over $c$ is the mean $\bar{x}$
- Penalizes large deviations more strongly
- Makes the update step (move centroid to mean) the provably optimal choice

### Is this NP-Hard?

Yes — finding the globally optimal partition is NP-hard in general. K-Means finds a **local minimum** efficiently using the iterative algorithm below.

---
## 3. Lloyd's Algorithm

The standard K-Means procedure (proposed by Stuart Lloyd in 1957, published 1982):

### Step 0 — Initialize
Choose $K$ initial centroids $\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_K$ (randomly or via K-Means++).

### Step 1 — Assignment (E-step)
Assign each point to the nearest centroid:

$$c_i = \arg\min_{k \in \{1,\dots,K\}} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

This partitions the data into $K$ Voronoi cells.

### Step 2 — Update (M-step)
Move each centroid to the mean of its assigned points:

$$\boldsymbol{\mu}_k \leftarrow \frac{1}{|C_k|} \sum_{\mathbf{x}_i : c_i = k} \mathbf{x}_i$$

### Step 3 — Repeat
Go back to Step 1. Stop when assignments no longer change (convergence).

### Complexity
- Each iteration: $O(nKd)$ where $d$ = number of features
- Typical convergence: 10–100 iterations
- Total: $O(nKdT)$ where $T$ = number of iterations

### Connection to EM Algorithm
Lloyd's algorithm is a **hard EM** (Expectation-Maximization):
- E-step: assign each point to exactly one cluster (hard assignment)
- M-step: update centroids to maximize likelihood (= minimize WCSS)

Gaussian Mixture Models (GMM) use **soft EM** with probabilistic assignments.

---
## 4. Mathematical Proof of Convergence

**Claim**: The inertia $J$ never increases between iterations.

**Assignment step**: For each point, we assign it to the closest centroid. This can only decrease or maintain $J$ since we never move a point farther from its centroid:
$$\|\mathbf{x}_i - \boldsymbol{\mu}_{c_i^{\text{new}}}\|^2 \leq \|\mathbf{x}_i - \boldsymbol{\mu}_{c_i^{\text{old}}}\|^2$$

**Update step**: The centroid that minimizes $\sum_{\mathbf{x}_i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}\|^2$ is the **mean** of the cluster. Any other value gives a higher sum:
$$\sum_{i} \|\mathbf{x}_i - \boldsymbol{\mu}\|^2 = \sum_{i} \|\mathbf{x}_i - \bar{\mathbf{x}}\|^2 + n\|\boldsymbol{\mu} - \bar{\mathbf{x}}\|^2 \geq \sum_{i} \|\mathbf{x}_i - \bar{\mathbf{x}}\|^2$$

**Conclusion**: Since $J$ is bounded below by 0 and decreases monotonically, and since there are finitely many partitions, the algorithm must converge in finite steps.

---
## 5. K-Means++ Initialization

Random initialization often leads to poor local minima. K-Means++ (Arthur & Vassilvitskii, 2007) uses a smarter scheme that **spreads initial centroids out**.

### Algorithm

1. Choose $\boldsymbol{\mu}_1$ uniformly at random from the data
2. For $k = 2, 3, \dots, K$:
   - For each point $\mathbf{x}_i$, compute:
     $$D(\mathbf{x}_i) = \min_{j < k} \|\mathbf{x}_i - \boldsymbol{\mu}_j\|^2$$
     (squared distance to nearest already-chosen centroid)
   - Choose next centroid $\boldsymbol{\mu}_k = \mathbf{x}_i$ with probability:
     $$P(\mathbf{x}_i) = \frac{D(\mathbf{x}_i)}{\sum_j D(\mathbf{x}_j)}$$
3. Run standard K-Means from these $K$ initial centroids

### Why It Works

Points far from all current centroids have high $D(\mathbf{x}_i)^2$ and thus high probability of being chosen next. This forces initial centroids to cover different parts of the space.

### Theoretical Guarantee

K-Means++ achieves expected inertia at most $O(\log K)$ times the optimal inertia — a significant improvement over random initialization.

In sklearn, `init='k-means++'` is the **default**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

np.random.seed(7)

# Generate 3-cluster data
centers_true = [[-3, -3], [0, 3], [3, -1]]
X = np.vstack([np.random.randn(80, 2) + c for c in centers_true])

# Visualize K-Means++ initialization
def kmeans_pp_init(X, K, seed=0):
    rng = np.random.default_rng(seed)
    n = len(X)
    centroids = [X[rng.integers(n)]]
    for _ in range(K - 1):
        D = np.array([min(np.sum((x - c)**2) for c in centroids) for x in X])
        probs = D / D.sum()
        idx = rng.choice(n, p=probs)
        centroids.append(X[idx])
    return np.array(centroids)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

for ax, title, init_fn in zip(
    axes,
    ['Random Init', 'K-Means++ Init'],
    [lambda X, K: X[np.random.choice(len(X), K, replace=False)],
     lambda X, K: kmeans_pp_init(X, K)]
):
    np.random.seed(42)
    init_centroids = init_fn(X, 3)
    ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=20, color='steelblue')
    for i, c in enumerate(init_centroids):
        ax.scatter(*c, s=300, marker='*', color=colors[i], zorder=5,
                   edgecolors='black', linewidth=1, label=f'Init centroid {i+1}')
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('K-Means++ spreads initial centroids across data', fontsize=13)
plt.tight_layout()
plt.savefig('kmeans_init_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. The Elbow Method

K-Means requires $K$ as input. The **elbow method** helps choose $K$ empirically.

### Procedure
1. Fit K-Means for $K = 1, 2, \dots, K_{\max}$
2. Record inertia $J(K)$ for each
3. Plot $J(K)$ vs $K$ — look for the "elbow" where the curve bends

### Why Does Inertia Always Decrease?

With $K+1$ clusters, we have strictly more flexibility than $K$ clusters. The worst case is splitting one cluster in half, which can only decrease WCSS.

### Reading the Elbow

The elbow is where adding more clusters gives diminishing returns in inertia reduction. Beyond the elbow, the clusters are being split unnecessarily.

**Note**: The elbow is sometimes unclear. Use the **silhouette score** (Section 04) as a more objective metric.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs

X_blobs, _ = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)
X_scaled = StandardScaler().fit_transform(X_blobs)

K_range = range(1, 13)
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# Compute second-derivative approximation for elbow detection
diffs = np.diff(inertias)
diffs2 = np.diff(diffs)
elbow_k = np.argmax(diffs2) + 2  # +2 because two diffs shift index

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-', lw=2)
plt.axvline(elbow_k, color='red', linestyle='--', label=f'Elbow at K={elbow_k}')
plt.xlabel('Number of Clusters K', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=13)
plt.xticks(K_range)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_elbow.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Detected elbow at K = {elbow_k}")

---
## 7. Limitations of K-Means

### 7.1 Assumes Spherical, Equal-Size Clusters

K-Means uses Euclidean distance to assign points. The Voronoi boundaries it creates are always **linear hyperplanes**. If clusters are:
- Elongated or non-convex → K-Means fails
- Very different sizes → K-Means breaks the large cluster
- Very different densities → K-Means ignores this

### 7.2 Must Specify K
K is a hyperparameter. Use elbow method or silhouette score to select it. Alternatively, use DBSCAN (no K required).

### 7.3 Sensitive to Scale
Feature with range [0, 1000] dominates feature with range [0, 1] in distance calculation. **Always standardize** before K-Means.

### 7.4 Sensitive to Outliers
Outliers pull centroids away from the true cluster centers. The mean is not robust. Use **K-Medoids** (PAM algorithm) which uses actual data points as centers.

### 7.5 Local Minimum
Different random initializations give different results. Run multiple times (`n_init=10` in sklearn) and keep the best.

In [ ]:
# Demonstrating K-Means failures
from sklearn.datasets import make_circles, make_moons
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [
    ('Non-spherical (Moons)', make_moons(300, noise=0.05, random_state=42), 2),
    ('Non-spherical (Circles)', make_circles(300, noise=0.04, factor=0.5, random_state=42), 2),
    ('Unequal sizes', (np.vstack([
        np.random.randn(250, 2) * 0.3 + [0, 0],
        np.random.randn(30, 2) * 0.3 + [4, 0],
        np.random.randn(30, 2) * 0.3 + [0, 4]
    ]), None), 3)
]

cmap = plt.cm.Set1
for ax, (title, (X_d, _), K) in zip(axes, datasets):
    X_s = StandardScaler().fit_transform(X_d)
    labels = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(X_s)
    ax.scatter(X_s[:, 0], X_s[:, 1], c=labels, cmap=cmap, s=20, alpha=0.8)
    ax.set_title(f'K-Means Failure: {title}', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cases Where K-Means Fails', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('kmeans_failures.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Implementation from Scratch

In [ ]:
import numpy as np

class KMeansScratch:
    def __init__(self, K, max_iter=300, tol=1e-4, init='kmeans++', n_init=10, random_state=None):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.n_init = n_init
        self.random_state = random_state

    def _init_centroids(self, X, rng):
        if self.init == 'random':
            idx = rng.choice(len(X), self.K, replace=False)
            return X[idx].copy()
        # K-Means++ initialization
        centroids = [X[rng.integers(len(X))]]
        for _ in range(self.K - 1):
            D2 = np.array([min(np.sum((x - c)**2) for c in centroids) for x in X])
            probs = D2 / D2.sum()
            centroids.append(X[rng.choice(len(X), p=probs)])
        return np.array(centroids)

    def _run_once(self, X, seed):
        rng = np.random.default_rng(seed)
        centroids = self._init_centroids(X, rng)

        for _ in range(self.max_iter):
            # Assignment step
            dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
            labels = dists.argmin(axis=1)

            # Update step
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if (labels == k).any() else centroids[k]
                for k in range(self.K)
            ])

            # Check convergence
            if np.linalg.norm(new_centroids - centroids) < self.tol:
                break
            centroids = new_centroids

        inertia = sum(
            np.sum((X[labels == k] - centroids[k])**2)
            for k in range(self.K) if (labels == k).any()
        )
        return labels, centroids, inertia

    def fit(self, X):
        best_inertia = np.inf
        rng = np.random.default_rng(self.random_state)
        seeds = rng.integers(0, 10000, self.n_init)
        for seed in seeds:
            labels, centroids, inertia = self._run_once(X, seed)
            if inertia < best_inertia:
                best_inertia = inertia
                self.labels_ = labels
                self.cluster_centers_ = centroids
                self.inertia_ = inertia
        return self

    def predict(self, X):
        dists = np.linalg.norm(X[:, None, :] - self.cluster_centers_[None, :, :], axis=2)
        return dists.argmin(axis=1)


# Test against sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

X_test, y_true = make_blobs(n_samples=300, centers=4, random_state=42)
X_test = StandardScaler().fit_transform(X_test)

km_scratch = KMeansScratch(K=4, random_state=42, n_init=10)
km_scratch.fit(X_test)

km_sk = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
km_sk.fit(X_test)

print(f"Scratch inertia:  {km_scratch.inertia_:.4f}")
print(f"sklearn inertia:  {km_sk.inertia_:.4f}")
print(f"Centroids close:  {np.allclose(np.sort(np.abs(km_scratch.cluster_centers_), axis=0),
                                        np.sort(np.abs(km_sk.cluster_centers_), axis=0), atol=0.1)}")

---
## 9. sklearn K-Means on Real Data

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score, silhouette_score
import matplotlib.pyplot as plt
import numpy as np

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

# Elbow + silhouette to choose K
K_range = range(2, 9)
inertias, sil_scores = [], []
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    labels = km.fit_predict(X_iris)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_iris, labels))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(K_range, inertias, 'bo-', lw=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].plot(K_range, sil_scores, 'ro-', lw=2)
axes[1].axvline(best_k, color='green', linestyle='--', label=f'Best K={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Final clustering with K=3 (known ground truth)
km_final = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42)
labels_final = km_final.fit_predict(X_iris)

X_2d = PCA(n_components=2).fit_transform(X_iris)
scatter = axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_final, cmap='Set1', s=30, alpha=0.8)
axes[2].set_title('K-Means (K=3) on Iris — PCA 2D view')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_iris.png', dpi=120, bbox_inches='tight')
plt.show()

ari = adjusted_rand_score(iris.target, labels_final)
sil = silhouette_score(X_iris, labels_final)
print(f"Adjusted Rand Index: {ari:.4f} (1.0 = perfect match with true labels)")
print(f"Silhouette Score:    {sil:.4f}")
print(f"Cluster sizes: {np.bincount(labels_final)}")

---
## 10. Summary

| Concept | Detail |
|---|---|
| Objective | Minimize WCSS / inertia $J = \sum_k \sum_{x \in C_k} \|x - \mu_k\|^2$ |
| Algorithm | Alternating assignment (nearest centroid) + update (move to mean) |
| Convergence | Guaranteed (J decreases monotonically), but only to local minimum |
| Initialization | K-Means++ spreads initial centroids, gives $O(\log K)$ approximation guarantee |
| Choosing K | Elbow method (inertia vs K) or silhouette score |
| Must scale? | **Yes** — always standardize features first |
| Cluster shapes | Only spherical/convex — use DBSCAN for arbitrary shapes |
| Outliers | Sensitive — use K-Medoids or DBSCAN instead |
| Time complexity | $O(nKdT)$ — very scalable |

### sklearn Quick Reference
```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
labels = km.fit_predict(X_scaled)
print(km.inertia_)          # WCSS
print(km.cluster_centers_)  # centroid coordinates
print(km.n_iter_)           # iterations until convergence
```